# RAG on Azure — Day 9: Reranking with the Semantic Ranker 

Here we make sure the **most** relevant one is at position #1 — by adding a second, more accurate model that reorders the top results.

- A two-stage retrieval pattern: **bi-encoder retrieval (fast, wide)** → **cross-encoder reranking (slow, precise)**.
- Azure AI Search **semantic ranker** is enabled on the index via a semantic configuration.
- One new helper, `retrieve_with_rerank()`, returns both the original retrieval score and the new `reranker_score` (0–4 range, cross-encoder relevance).
- A side-by-side demo where hybrid alone gets the right chunk into the top 10 but not at #1, and the reranker promotes it to #1.


- Azure AI Search service must be on a tier that supports semantic ranker -  use Basic or Standard.
- Semantic ranker has a separate transaction quota (1000/month free on most regions)

## 0. Install dependencies

In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Setup

In [2]:
import sys
sys.path.insert(0, "..")

from common import load_config, get_openai_client, get_search_client, get_search_index_client
from common import chunk_text, load_pdfs, extract_sections
from common import embed_texts, retrieve_hybrid, retrieve_with_rerank
from common import answer

config = load_config()
openai = get_openai_client(config)
index_client = get_search_index_client(config)
INDEX_NAME = "rag-techdocs-day9"
print("Setup OK. Index:", INDEX_NAME)

Setup OK. Index: rag-techdocs-day9


## 2. Load, chunk, embed

In [3]:
from pathlib import Path
import re

DATA_DIR = Path("../data-day8")
print("Looking in:", DATA_DIR.resolve())
raw_docs = load_pdfs(DATA_DIR)
assert raw_docs, f"No PDFs in {DATA_DIR.resolve()}"

records = []
for d in raw_docs:
    sections = extract_sections(d["text"]) or [("Body", d["text"])]
    for section_title, section_body in sections:
        safe_src = re.sub(r"[^A-Za-z0-9_-]", "_", d["source"])
        safe_sec = re.sub(r"[^A-Za-z0-9_-]", "_", section_title)[:30]
        for i, chunk in enumerate(chunk_text(section_body)):
            records.append({
                "id":      f"{safe_src}-{safe_sec}-{i}",
                "content": chunk,
                "source":  d["source"],
                "section": section_title,
            })

print(f"{len(records)} chunks from {len(raw_docs)} PDFs.")
chunk_vectors = embed_texts(openai, [r["content"] for r in records], config["embedding_deployment"])
EMBED_DIM = len(chunk_vectors[0])
print(f"Embedded {len(chunk_vectors)} chunks. Dim: {EMBED_DIM}")

Looking in: C:\Users\sunil\OneDrive\Documents\GitHub\Basic RAG\RAG\data-day8
13 chunks from 5 PDFs.
Embedded 13 chunks. Dim: 3072


## 3. Create the index — now with a semantic configuration

The new piece is the `SemanticConfiguration`. It tells the cross-encoder which fields to read when scoring relevance: the `section` name acts as a "title", and `content` is the body. With this config attached to the index, query-time reranking is a one-flag opt-in.

In [4]:
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField, SearchFieldDataType,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
    SemanticConfiguration, SemanticPrioritizedFields, SemanticField, SemanticSearch,
)

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SearchableField(name="section", type=SearchFieldDataType.String, filterable=True),
    SimpleField(name="source", type=SearchFieldDataType.String, filterable=True),
    SearchField(
        name="contentVector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBED_DIM,
        vector_search_profile_name="vprofile",
    ),
]
vs = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw")],
    profiles=[VectorSearchProfile(name="vprofile", algorithm_configuration_name="hnsw")],
)

# The semantic configuration drives reranking.
semantic_config = SemanticConfiguration(
    name="default-semantic",
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name="section"),
        content_fields=[SemanticField(field_name="content")],
    ),
)
semantic_search = SemanticSearch(configurations=[semantic_config])

try:
    index_client.delete_index(INDEX_NAME)
    print(f"Deleted existing index '{INDEX_NAME}'.")
except Exception as e:
    print("No existing index (ok):", type(e).__name__)

index_client.create_index(SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vs,
    semantic_search=semantic_search,
))
print(f"Index '{INDEX_NAME}' created with semantic configuration 'default-semantic'.")

Deleted existing index 'rag-techdocs-day9'.
Index 'rag-techdocs-day9' created with semantic configuration 'default-semantic'.


## 4. Upload

In [5]:
search = get_search_client(config, INDEX_NAME)
docs = [{**r, "contentVector": v} for r, v in zip(records, chunk_vectors)]
for i in range(0, len(docs), 100):
    search.upload_documents(documents=docs[i:i+100])
print(f"Uploaded {len(docs)} chunks.")

Uploaded 13 chunks.


## 5. Hybrid vs hybrid + reranking — side by side

The reranker reads the *original question* and *each candidate chunk* together and assigns a relevance score from 0 to 4 (Azure's documented range). A score around 2.5+ generally means "yes, this is relevant"; under 1 means "no, probably not."

Watch the position changes — the same passages, reordered.

In [6]:
SELECT = ["content", "source", "section"]

def compare(query, k=5):
    print(f"\nQuery: {query!r}")
    print("=" * 78)

    hyb = retrieve_hybrid(
        search, openai, query, config["embedding_deployment"],
        k=k, select=SELECT,
    )
    rrk = retrieve_with_rerank(
        search, openai, query, config["embedding_deployment"],
        k=k, select=SELECT,
    )

    print(f"\n  -- HYBRID ONLY (top {k}) --")
    for i, h in enumerate(hyb, 1):
        print(f"   {i}. [{h['score']:.3f}] {h['source']} - {h['section']}")
        print(f"        {h['content'][:90]}{'...' if len(h['content'])>90 else ''}")

    print(f"\n  -- HYBRID + SEMANTIC RERANK (top {k}) --")
    for i, h in enumerate(rrk, 1):
        print(f"   {i}. [orig {h['score']:.3f} | rerank {h['reranker_score']:.2f}] "
              f"{h['source']} - {h['section']}")
        print(f"        {h['content'][:90]}{'...' if len(h['content'])>90 else ''}")

## 6. Demo A — a question where reranking changes the order

This kind of question is the reranker's strength: a paraphrased, conversational ask that maps to a *specific* error code without naming it. Hybrid finds plausible candidates; the cross-encoder picks the right one.

In [7]:
compare("My subscription got cut off because I didn't renew. What error will the API return?")


Query: "My subscription got cut off because I didn't renew. What error will the API return?"

  -- HYBRID ONLY (top 5) --
   1. [0.033] 02_Error_Codes.pdf - Introduction
        Error Code Reference
Version 3.7 | All ERR-XXXX codes returned by the API
Authentication e...
   2. [0.033] 01_API_Reference.pdf - Subscriptions
        Endpoint: POST /v2/accounts/{accountId}/subscriptions
Creates a new subscription on the ac...
   3. [0.032] 02_Error_Codes.pdf - Introduction
        enticated principal does not have the required role
to perform this operation. Required ro...
   4. [0.031] 01_API_Reference.pdf - Rate Limits
        Default rate limit is 100 requests per minute per API key. Enterprise tier (SKU
AC-CORE-EN...
   5. [0.030] 01_API_Reference.pdf - Accounts
        Endpoint: GET /v2/accounts/{accountId}
Retrieves account details including subscription st...

  -- HYBRID + SEMANTIC RERANK (top 5) --
   1. [orig 0.033 | rerank 2.35] 02_Error_Codes.pdf - Introduction
        Error Co

## 7. Demo B — a question that needs cross-document reading

Reranking shines when *understanding the question and the passage together* is required. Notice how the reranker_score discriminates sharply between "almost right" and "exactly right" chunks even when their original BM25/vector scores are close.

In [8]:
compare("Which plan do I need if my team needs more than 100 API requests per minute?")


Query: 'Which plan do I need if my team needs more than 100 API requests per minute?'

  -- HYBRID ONLY (top 5) --
   1. [0.033] 01_API_Reference.pdf - Rate Limits
        Default rate limit is 100 requests per minute per API key. Enterprise tier (SKU
AC-CORE-EN...
   2. [0.033] 05_Pricing_SKU_Guide.pdf - Introduction
        Pricing & SKU Guide
Effective 2024-10-01 | All prices in USD
Core platform SKUs
AC-CORE-ST...
   3. [0.032] 02_Error_Codes.pdf - Introduction
        enticated principal does not have the required role
to perform this operation. Required ro...
   4. [0.032] 05_Pricing_SKU_Guide.pdf - Introduction
        -ENT-ANNUAL — Analytics module on the Enterprise tier. $2999 per user per year.
Adds every...
   5. [0.030] 01_API_Reference.pdf - Authentication
        All requests require a Bearer token in the Authorization header. Obtain a token via the /v...

  -- HYBRID + SEMANTIC RERANK (top 5) --
   1. [orig 0.033 | rerank 2.59] 05_Pricing_SKU_Guide.pdf - Introduction
  

## 8. Demo C — vague phrasing, no exact terms

Hybrid retrieval has nothing keyword-y to grip onto. The reranker uses understanding, not lexical match.

In [9]:
compare("How do I sign out of the command line tool on my machine?")


Query: 'How do I sign out of the command line tool on my machine?'

  -- HYBRID ONLY (top 5) --
   1. [0.033] 04_CLI_Reference.pdf - Introduction
        acme-cli — Command Line Reference
Companion CLI for Acme Cloud Platform
Authentication
acm...
   2. [0.032] 04_CLI_Reference.pdf - Introduction
        ams the latest hour of platform logs.
acme-cli logs tail --filter error-code=ERR-4012 — fi...
   3. [0.031] 03_Release_Notes_v3_7.pdf - Introduction
        Release Notes — Acme Cloud Platform
Release v3.7 series
v3.7.0 — Major release (released 2...
   4. [0.030] 01_API_Reference.pdf - Authentication
        All requests require a Bearer token in the Authorization header. Obtain a token via the /v...
   5. [0.017] 02_Error_Codes.pdf - Introduction
        Error Code Reference
Version 3.7 | All ERR-XXXX codes returned by the API
Authentication e...

  -- HYBRID + SEMANTIC RERANK (top 5) --
   1. [orig 0.033 | rerank 2.01] 04_CLI_Reference.pdf - Introduction
        acme-cli — Command 

## 9. End-to-end answer using reranked top 5

Generation is unchanged — just swap `retrieve_hybrid` for `retrieve_with_rerank` and the LLM sees the best possible context.

In [10]:
def reranked_answer(question, k=5):
    hits = retrieve_with_rerank(
        search, openai, question, config["embedding_deployment"],
        k=k, select=SELECT,
    )
    return answer(openai, config["chat_deployment"], question, hits, cite=True), hits

for q in [
    "My subscription got cut off because I didn't renew. What error will the API return?",
    "Which plan do I need for more than 100 API requests per minute?",
    "How do I sign out of the command line tool?",
]:
    ans, hits = reranked_answer(q)
    print(f"\nQ: {q}")
    print(f"A: {ans}")
    print(f"  top reranker scores: {[round(h['reranker_score'], 2) for h in hits]}")
    print("-" * 78)


Q: My subscription got cut off because I didn't renew. What error will the API return?
A: If your subscription entitlement has expired and you attempt to perform write operations on the account, the API will return **ERR-4012**. This error indicates that the subscription entitlement has lapsed, and the account must be renewed before further write operations are accepted. However, read-only endpoints will continue to work for a 30-day grace period [1], [2].
  top reranker scores: [2.35, 2.31, 2.14, 2.11, 2.08]
------------------------------------------------------------------------------

Q: Which plan do I need for more than 100 API requests per minute?
A: To exceed 100 API requests per minute, you need the **Enterprise tier (AC-CORE-ENT-ANNUAL)**, which raises the rate limit to 1000 requests per minute [1], [2].
  top reranker scores: [2.64, 2.54, 1.72, 1.61, 1.45]
------------------------------------------------------------------------------

Q: How do I sign out of the command line

## Day 9 checklist

1. Index has a `SemanticConfiguration` named `default-semantic`
2. `retrieve_with_rerank()` returns both `score` and `reranker_score`
3. On a paraphrased question (Demo A), the reranker promotes the truly-relevant chunk above superficially-similar ones
4. Reranker scores meaningfully discriminate: the top hit has a noticeably higher reranker_score than the rest
5. End-to-end answers are visibly better grounded than Day 8

### Learnings
- **Bi-encoder vs cross-encoder.** Retrieval uses a bi-encoder (encode query and docs separately, compare by similarity) — fast at any scale. Reranking uses a cross-encoder (encode query *with* each candidate together) — slow per query, but vastly more accurate. They're complementary: bi-encoder for recall over millions, cross-encoder for precision over dozens.
- **Reranking is the highest-precision boost per dollar.** No new infrastructure (it's built into Azure AI Search), no embedding cost, no LLM call. Just a `query_type="semantic"` flag.
- **Cost/latency tradeoff.** Reranking adds ~100–300ms per query and consumes a "semantic search transaction" against your quota. For chat workloads this is usually a clear win; for very-high-throughput pipelines you may sample-rerank.